# Лабораторна робота №3. Розпізнавання тексту з застосуванням ШІ

**Студент:** Скиданчук М.Д.  
**Варіант:** 6  
**Набір:** Perestoroha OCR · **Інструменти:** EasyOCR, PaddleOCR · **Чинник:** зниження роздільності

## 0. Налаштування і персональні параметри

In [1]:
import os, re, time, hashlib, unicodedata, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image, ImageDraw, ImageFont
from rapidfuzz.distance import Levenshtein
from scipy.stats import spearmanr
warnings.filterwarnings("ignore")

# ---- персональні параметри ----
STUDENT = "Скиданчук М.Д."
VARIANT = 6
STUDENT_ID = 962
SEED = 1000 * VARIANT + STUDENT_ID   # 6962
rng = np.random.default_rng(SEED)

os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)

def stamp(ax, title):
    """Заголовок графіка з персональним підписом студента."""
    ax.set_title(f"{title}\n{STUDENT}, варіант {VARIANT}, seed={SEED}", fontsize=10)

print(STUDENT, VARIANT, SEED)

Скиданчук М.Д. 6 6962


## 1. Гіпотези (записано до запуску розпізнавання, 07.11.2026)

Прогнози сформульовано до першого запуску EasyOCR і PaddleOCR на даних. Чернетку підготовлено разом з ШІ-асистентом, студент її розглянув і погодив; оцінка («підтверджено / спростовано») буде додана після експериментів.

| № | Гіпотеза (прогноз) | Обґрунтування | Оцінка |
|---|---|---|---|
| Г1 | На картках Perestoroha без зниження роздільності (×1) CER обох інструментів не менший за 0,60; PaddleOCR не буде кращим за EasyOCR. | Інструменти навчені на друкованому тексті, а картки рукописні зі старими літерами (ѣ, ъ, титла). | — |
| Г2 | При зниженні роздільності ×4 CER обох інструментів не менший за 0,90; найстрімкіше зростання між ×1 і ×2,5. Поріг 10% перевищується вже на ×1 (плато одразу високо). | При ×4 картка має ~290×188 пікселів, штрихи рукопису зливаються. | — |
| Г3 | Розширена нормалізація історичних графем (ѣ→і, ъ→∅, ѡ→о тощо, без титл і наголосів) знизить CER лише на 2–5 процентних пунктів. | Основна помилка пов'язана з рукописом, а не з окремими старими літерами. | — |
| Г4 | На фото меморіальної дошки / старого напису CER буде у 2–3 рази нижчим, ніж на картках Perestoroha. | Друкований або вирізаний текст ближчий до даних, на яких навчені моделі. | — |
| Г5 | На фото власної рукописної сторінки CER буде щонайменше у 10 разів вищим за CER на синтетичному друкованому тексті (його очікую ≤ 0,05 на ×1). | Рукопис, тіні й кут зйомки (domain gap). | — |
| Г6 | Бінаризація Оцу погіршить CER обох інструментів не менш ніж на 5 п.п.; збільшення ×2 з медіанним фільтром покращить результат. | Нейромережі навчені на зображеннях у відтінках сірого; бінаризація руйнує тонкі штрихи. | — |
| Г7 | На CPU PaddleOCR опрацьовує зображення швидше за EasyOCR (час ≤ 0,7 від часу EasyOCR). | Легший конвеєр детекції й розпізнавання. | — |


## 2. Відкритий набір Perestoroha OCR: огляд структури

In [2]:
from datasets import load_dataset

ds = load_dataset("lang-uk/perestoroha-ocr")
print(ds)                       # спліти, кількість записів, назви полів
for name, part in ds.items():
    print(name, part.features)


Generating test split: 100%|██████████| 30/30 [00:00<00:00, 1317.42 examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'image', 'transcription', 'card_num_primary', 'card_num_secondary', 'card_num_tertiary', 'card_num_notes', 'source_city', 'source_date', 'source_reference', 'notes', 'reviewed_at'],
        num_rows: 278
    })
    validation: Dataset({
        features: ['id', 'image', 'transcription', 'card_num_primary', 'card_num_secondary', 'card_num_tertiary', 'card_num_notes', 'source_city', 'source_date', 'source_reference', 'notes', 'reviewed_at'],
        num_rows: 122
    })
    test: Dataset({
        features: ['id', 'image', 'transcription', 'card_num_primary', 'card_num_secondary', 'card_num_tertiary', 'card_num_notes', 'source_city', 'source_date', 'source_reference', 'notes', 'reviewed_at'],
        num_rows: 30
    })
})
train {'id': Value('string'), 'image': Image(mode=None, decode=True), 'transcription': Value('string'), 'card_num_primary': Value('string'), 'card_num_secondary': Value('string'), 'card_num_tertiary': Value('s

In [3]:
# перший запис: що лежить у кожному полі
split = list(ds.keys())[0]
sample = ds[split][0]
for k, v in sample.items():
    if hasattr(v, "size"):                     # зображення PIL
        print(k, "-> image", v.size, v.mode)
    else:
        print(k, "->", repr(v)[:300])
sample[[k for k, v in sample.items() if hasattr(v, "size")][0]].save("data/perestoroha_0.png")


id -> 'Auto-Color0002/182'
image -> image (1160, 752) RGB
transcription -> 'пан воєвода кієвскій для коней\r\nєго себѣ слалъ с килькома сты червоных\r\nЛьвів, 1605-1606, Перест. 38'
card_num_primary -> '10'
card_num_secondary -> None
card_num_tertiary -> '153'
card_num_notes -> '10 top-left in pencil (circled); 153 bottom-right in pencil'
source_city -> 'Львів'
source_date -> '1605-1606'
source_reference -> 'Перест. 38'
notes -> "Second line opens with 'ето' as written (likely scribe's form of 'єго'); 'сты' written with titlo-like horizontal bar above but treated as plain letters per preset rules."
reviewed_at -> '2026-05-06T18:54:16.666799+00:00'
